# Gate 1: generate candidates with a second model (GPU step)

This notebook only **generates** code. Labelling, pool building and all evaluation run on the laptop (CPU).

**Settings:** Accelerator **GPU T4 x1**, Internet **on**. For a private repo, add a Kaggle secret named `GITHUB_TOKEN` (*Add-ons → Secrets*).

**Then:** *Save Version → Save & Run All*. When it finishes, download `gen_<model>.zip` from the Output tab.

Prompt, seeds and decoding settings are the same as the Baseline 1 run (prompt hash `ebbe8bceae71`).

**New datasets (taxi, titanic):** run it **twice**, once with `MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"` and once with `"Qwen/Qwen2.5-1.5B-Instruct"`, so the new tasks get the same two generators as pool v2.

In [1]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
N_SAMPLES = 5
TEMPERATURE = 0.7
LIMIT = None            # e.g. 3 for a quick dry run
DATASETS = ["taxi", "titanic"]   # None = all tasks; pools v1/v2 already cover adult and airbnb
TAG = MODEL_ID.split("/")[-1].lower().replace(".", "")
OUT = f"/kaggle/working/gen_{TAG}" + ("_" + "_".join(DATASETS) if DATASETS else "")
print(TAG, OUT)

qwen25-15b-instruct /kaggle/working/gen_qwen25-15b-instruct_taxi_titanic


In [2]:
# ---------------- Get the code (no pip install: the kernel's torch/transformers are used as is) ----------------
import os, shutil, subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)        # always a fresh clone of the latest push
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s (%cd)"], capture_output=True, text=True).stdout)

NEEDED = ["src/igt/__init__.py", "src/igt/agents/__init__.py", "src/igt/agents/code_generator.py",
          "eval/generate.py", "eval/prompts/schema_b1-v1.json", "eval/tasks"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"These files are not in the GitHub repo (copy them from the workspace and push): {missing}"
print("all needed files present")

no GITHUB_TOKEN secret; cloning without a token (public repo)


Cloning into '/kaggle/working/repo'...


efc5963ebf78c083fdff6a438d7d50213a3a2ab2 agent eval (Fri Oct 30 14:04:03 2026 +0530)

all needed files present


In [3]:
# ---------------- Check the GPU and the prompt ----------------
import sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator")
sys.path.insert(0, "src")
from igt.agents.code_generator import PROMPT_HASH
assert PROMPT_HASH == "ebbe8bceae71", PROMPT_HASH
print("prompt hash OK:", PROMPT_HASH)

GPU: Tesla T4
prompt hash OK: ebbe8bceae71


In [4]:
# ---------------- Generate (resumable) ----------------
cmd = ["python", "eval/generate.py", "--model", MODEL_ID, "--out", OUT,
       "--n-samples", str(N_SAMPLES), "--temperature", str(TEMPERATURE)]
if DATASETS:
    cmd += ["--datasets", *DATASETS]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-3000:])
print(p.stderr[-3000:])
assert p.returncode == 0

greedy 8/60 (7s)
greedy 16/60 (12s)
greedy 24/60 (19s)
greedy 32/60 (27s)
greedy 40/60 (34s)
greedy 48/60 (37s)
greedy 56/60 (41s)
greedy 60/60 (47s)
sampled 1/60 (55s)
sampled 2/60 (63s)
sampled 3/60 (67s)
sampled 4/60 (71s)
sampled 5/60 (76s)
sampled 6/60 (80s)
sampled 7/60 (88s)
sampled 8/60 (95s)
sampled 9/60 (102s)
sampled 10/60 (105s)
sampled 11/60 (117s)
sampled 12/60 (121s)
sampled 13/60 (130s)
sampled 14/60 (141s)
sampled 15/60 (149s)
sampled 16/60 (156s)
sampled 17/60 (167s)
sampled 18/60 (176s)
sampled 19/60 (180s)
sampled 20/60 (183s)
sampled 21/60 (187s)
sampled 22/60 (197s)
sampled 23/60 (202s)
sampled 24/60 (207s)
sampled 25/60 (215s)
sampled 26/60 (223s)
sampled 27/60 (227s)
sampled 28/60 (232s)
sampled 29/60 (237s)
sampled 30/60 (258s)
sampled 31/60 (266s)
sampled 32/60 (270s)
sampled 33/60 (276s)
sampled 34/60 (285s)
sampled 35/60 (292s)
sampled 36/60 (295s)
sampled 37/60 (304s)
sampled 38/60 (310s)
sampled 39/60 (321s)
sampled 40/60 (326s)
sampled 41/60 (330s)
sample

In [5]:
# ---------------- Quick look and zip ----------------
import json, shutil, collections
G = [json.loads(l) for l in open(f"{OUT}/generations.jsonl")]
print("generations:", len(G), "| truncated:", sum(g["truncated"] for g in G))
print("per split:", collections.Counter(g["split"] for g in G))
print(G[0]["raw_output"][:500])
open(f"{OUT}/git_commit.txt", "w").write(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("saved:", shutil.make_archive(OUT, "zip", OUT))
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)   # keep only the outputs

generations: 360 | truncated: 0
per split: Counter({'test': 300, 'dev': 60})
```python
import pandas as pd

def transform(df):
    df['fare_plus_tip'] = df['fare_amount'] + df['tip_amount']
    return df
```
saved: /kaggle/working/gen_qwen25-15b-instruct_taxi_titanic.zip
